In [4]:
import os
import re
import time
import shutil
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from datetime import datetime, timedelta
from typing import List


In [5]:
# Dynamically resolve the user home directory so the notebook runs on any machine.
# All source file paths are declared in a single dictionary for easy maintenance.
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"SharePoint root folder not found: {test_path}")

folder_paths = {
    # HC Master Database is the single source of truth for all sheets below
    "hc_staffing":        f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx",
    "hc_active":          f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx",
    "hc_inactive":        f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx",
    "date_format":        f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Master Database - 2026.xlsx",
    # Output folder for monthly CSVs
    "hc_extend_by_month": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month/",
    # Static resource directory
    "resources":          f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources",
}


In [6]:
# ---------------------------------------------------------------------------
# Helper: read one sheet from the Workday dump, locate the real header row,
# and keep only rows where msa_col contains "expedia" (case-insensitive).
# Uses Polars + calamine engine for maximum speed.
# ---------------------------------------------------------------------------
def extract_and_clean_sheet(
    file_path: Path,
    sheet_name: str,
    header_col_name: str,
    msa_col: str,
) -> pl.DataFrame:
    # Read without a header so we can locate the true header row dynamically
    df_raw = pl.read_excel(
        source=file_path,
        sheet_name=sheet_name,
        engine="calamine",
        read_options={"header_row": None},
    )

    # Find the row whose first column matches the expected header name
    first_col = df_raw.columns[0]
    first_col_values = df_raw[first_col].cast(pl.Utf8).to_list()
    header_indices = [
        i for i, val in enumerate(first_col_values)
        if val is not None and str(val).strip() == header_col_name
    ]
    header_idx = header_indices[0] if header_indices else 0

    # Slice from that row onward; use the first row as column names
    df_data = df_raw.slice(header_idx)
    headers = df_data.row(0)
    new_cols = [
        str(h).strip() if h is not None else f"unnamed_{i}"
        for i, h in enumerate(headers)
    ]
    df_cleaned = df_data.slice(1)
    df_cleaned.columns = new_cols

    # Cast the ID column to Int64 (serial numbers)
    id_col = new_cols[0]
    df_cleaned = df_cleaned.with_columns(pl.col(id_col).cast(pl.Int64, strict=False))

    # Filter to Expedia rows only
    df_filtered = df_cleaned.filter(
        pl.col(msa_col).str.to_lowercase().str.contains("expedia")
    )
    return df_filtered


# ---------------------------------------------------------------------------
# Workday dump → monthly WD_{YYYY_MM}.xlsx files
# Each file contains three sheets: Employee Master, Termination, Transfer.
# ---------------------------------------------------------------------------
def process_workday_headcount(input_file: str, output_dir: str) -> None:
    file_path = Path(input_file)
    out_dir   = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Reading Workday Dump: {file_path.name} ...")

    # 1. Employee Master
    df_master = extract_and_clean_sheet(
        file_path, sheet_name="Employee Master",
        header_col_name="EMPLOYEE_NUMBER", msa_col="MSA Client",
    )

    # 2. Termination — parse date and truncate to month-start
    df_term = extract_and_clean_sheet(
        file_path, sheet_name="Termination",
        header_col_name="EMPLOYEE_ID", msa_col="MSA",
    )
    df_term = df_term.with_columns(
        pl.col("TERMINATION DATE")
          .cast(pl.Utf8).str.slice(0, 10)
          .str.to_date("%Y-%m-%d", strict=False)
          .alias("Parsed_Date")
    ).with_columns(
        pl.col("Parsed_Date").dt.truncate("1mo").alias("Month_Start")
    )

    # 3. Transfer — same date treatment
    df_trans = extract_and_clean_sheet(
        file_path, sheet_name="Transfer",
        header_col_name="Employee ID", msa_col="MSA Client Current",
    )
    df_trans = df_trans.with_columns(
        pl.col("Effective Date")
          .cast(pl.Utf8).str.slice(0, 10)
          .str.to_date("%Y-%m-%d", strict=False)
          .alias("Parsed_Date")
    ).with_columns(
        pl.col("Parsed_Date").dt.truncate("1mo").alias("Month_Start")
    )

    # 4. Collect all distinct months present in either Termination or Transfer
    months_term  = df_term.select("Month_Start").drop_nulls().unique().to_series().to_list()
    months_trans = df_trans.select("Month_Start").drop_nulls().unique().to_series().to_list()
    all_months   = sorted(set(months_term + months_trans))
    print(f"Found {len(all_months)} distinct months. Generating files ...")

    # 5. Write one .xlsx per month (Employee Master is repeated in every file)
    for month in all_months:
        month_str   = month.strftime("%Y_%m")
        out_filename = out_dir / f"WD_{month_str}.xlsx"

        df_term_m  = df_term.filter(pl.col("Month_Start") == month).drop(["Parsed_Date", "Month_Start"])
        df_trans_m = df_trans.filter(pl.col("Month_Start") == month).drop(["Parsed_Date", "Month_Start"])

        with pd.ExcelWriter(out_filename, engine="xlsxwriter") as writer:
            df_master.to_pandas().to_excel(writer, sheet_name="Employee Master", index=False)
            df_term_m.to_pandas().to_excel(writer,  sheet_name="Termination",     index=False)
            df_trans_m.to_pandas().to_excel(writer, sheet_name="Transfer",        index=False)

        print(f"  Generated: {out_filename.name}")

    print("All monthly files generated successfully.")


if __name__ == "__main__":
    INPUT_FILE       = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/Workday Dump_Till_Date.xlsb"
    OUTPUT_DIRECTORY = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/WD"
    process_workday_headcount(INPUT_FILE, OUTPUT_DIRECTORY)


# ---------------------------------------------------------------------------
# Generate a single Combined_WD_Data.csv that merges Employee Master +
# Termination into one flat file for downstream reporting.
# ---------------------------------------------------------------------------
def generate_combined_headcount_csv() -> None:
    input_file   = Path(f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/Workday Dump_Till_Date.xlsb")
    test_out_dir = Path(f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/")
    test_out_dir.mkdir(parents=True, exist_ok=True)
    output_csv = test_out_dir / "Combined_WD_Data.csv"

    df_master = extract_and_clean_sheet(
        input_file, sheet_name="Employee Master",
        header_col_name="EMPLOYEE_NUMBER", msa_col="MSA Client",
    ).select([
        "EMPLOYEE_NUMBER", "FULL_NAME", "ORIGINAL_DATE_OF_HIRE", "WORKER_CATEGORY",
        "Email - Work", "Job Title", "Compensation Grade", "SUPERVISOR_ID",
        "SUPERVISOR_FULL_NAME", "SUPERVISOR_EMAIL_ID", "Employee Status",
        "Contract End Date", "Fixed Term Hire End Date",
        "MANAGER_02_ID", "MANAGER_02_FULL_NAME", "MANAGER_02_EMAIL_ID",
        "JOB_FUNCTION_DESCRIPTION",
    ])

    df_term = extract_and_clean_sheet(
        input_file, sheet_name="Termination",
        header_col_name="EMPLOYEE_ID", msa_col="MSA",
    ).select([
        pl.col("EMPLOYEE_ID").alias("EMPLOYEE_NUMBER"),
        pl.col("FULL_NAME"),
        pl.col("EMAIL_ADDRESS").alias("Email - Work"),
        pl.col("ORIGINAL_HIRE_DATE").alias("ORIGINAL_DATE_OF_HIRE"),
        "END EMPLOYMENT DATE", "Termination Date", "Eligible for Rehire",
        "LWD", "TERMINATION DATE", "Termination Reason", "Resignation Reason",
        "WORKER_CATEGORY",
        pl.col("JOB_TITLE").alias("Job Title"),
        "Compensation Grade", "SUPERVISOR_ID", "SUPERVISOR_FULL_NAME",
        "SUPERVISOR_EMAIL_ID",
        pl.col("EMPLOYEE STATUS").alias("Employee Status"),
        "Contract End Date",
        pl.col("JOB_FUNCTION").alias("JOB_FUNCTION_DESCRIPTION"),
    ])

    # Diagonal concat handles columns present in one frame but not the other
    df_combined = pl.concat([df_master, df_term], how="diagonal")

    # Parse all date columns uniformly (first 10 chars = YYYY-MM-DD)
    date_cols = [
        "ORIGINAL_DATE_OF_HIRE", "END EMPLOYMENT DATE", "LWD",
        "Termination Date", "TERMINATION DATE", "Contract End Date",
        "Fixed Term Hire End Date",
    ]
    for col_name in date_cols:
        if col_name in df_combined.columns:
            df_combined = df_combined.with_columns(
                pl.col(col_name)
                  .cast(pl.Utf8).str.slice(0, 10)
                  .str.to_date("%Y-%m-%d", strict=False)
            )

    df_combined = df_combined.with_columns([
        pl.col("EMPLOYEE_NUMBER").cast(pl.Int64, strict=False),
        pl.col("SUPERVISOR_ID").cast(pl.Int64, strict=False),
        pl.col("MANAGER_02_ID").cast(pl.Int64, strict=False),
    ])

    df_combined.write_csv(output_csv)
    print(f"Combined CSV written to: {output_csv}")


if __name__ == "__main__":
    generate_combined_headcount_csv()


Reading Workday Dump: Workday Dump_Till_Date.xlsb ...
Found 31 distinct months. Generating files ...
  Generated: WD_2023_11.xlsx
  Generated: WD_2023_12.xlsx
  Generated: WD_2024_01.xlsx
  Generated: WD_2024_02.xlsx
  Generated: WD_2024_03.xlsx
  Generated: WD_2024_04.xlsx
  Generated: WD_2024_05.xlsx
  Generated: WD_2024_06.xlsx
  Generated: WD_2024_07.xlsx
  Generated: WD_2024_08.xlsx
  Generated: WD_2024_09.xlsx
  Generated: WD_2024_10.xlsx
  Generated: WD_2024_11.xlsx
  Generated: WD_2024_12.xlsx
  Generated: WD_2025_01.xlsx
  Generated: WD_2025_02.xlsx
  Generated: WD_2025_03.xlsx
  Generated: WD_2025_04.xlsx
  Generated: WD_2025_05.xlsx
  Generated: WD_2025_06.xlsx
  Generated: WD_2025_07.xlsx
  Generated: WD_2025_08.xlsx
  Generated: WD_2025_09.xlsx
  Generated: WD_2025_10.xlsx
  Generated: WD_2025_11.xlsx
  Generated: WD_2025_12.xlsx
  Generated: WD_2026_01.xlsx
  Generated: WD_2026_02.xlsx
  Generated: WD_2026_03.xlsx
  Generated: WD_2026_04.xlsx
  Generated: WD_2026_05.xlsx


In [109]:
# ---------------------------------------------------------------------------
# Read the four key sheets from HC Master Database (Extract step).
# Then cross-join employees x date range to create one row per employee per day.
# This 'expanded' table is the backbone of the dynamic headcount dashboard.
#
# BUG FIXES applied vs. original:
#   1. 'CSG Joining Data' (typo in Inactive sheet) -> selected as-is, renamed after.
#   2. 'Manager Name' in Inactive renamed to 'Manager/OM Name' to align with Active.
#   3. Trailing-space columns in Inactive ('Lodging - Training start date ',
#      'Non-Lodging - Training start date ') selected with exact names, then renamed.
#   4. for-loop inline assignment was a SyntaxError -- split onto two lines.
#   5. Cross-join rewritten with Polars for speed; pandas only for the merge.
# ---------------------------------------------------------------------------

# 1. Read all required sheets from HC Master Database
staffing_records = pd.read_excel(folder_paths["hc_staffing"], sheet_name="Staffing_Records")
hc_active        = pd.read_excel(folder_paths["hc_active"],   sheet_name="Active")
hc_inactive      = pd.read_excel(folder_paths["hc_inactive"], sheet_name="Inactive")
date_format      = pd.read_excel(folder_paths["date_format"], sheet_name="Date_Format")

# 2. Select columns needed from the Active sheet
hc_active = hc_active[[
    "Site", "Queue Group", "CSG Joining Date", "Location", "People ID",
    "IEX ID", "OracleID", "Employee Name", "Alias", "Gender",
    "Contract Start Date", "Contract End Date", "Designation", "Grade", "LOB",
    "Role", "Multiple Chat Effective Date", "Primary role",
    "Secondary role", "TL ID", "Supervisor Name", "Supervisor Email",
    "Manager/OM Name", "Email Id", "Wave", "CCT Training",
    "Lodging - Training start date", "Lodging - Training end date",
    "Lodging - Nesting Date", "Lodging - Certification Date (Original)",
    "Lodging - Certification Date (Actual)", "Lodging - Certification status",
    "Non-Lodging - Training start date", "Non-Lodging - Training end date",
    "Non-Lodging - Nesting Date", "Non-Lodging - Certification Date",
    "Non-Lodging - Certification Date (Actual)", "Non-Lodging - Certification status",
    "Production Date", "AON", "Tenure", "Nationality", "MSA", "Current Console Queue",
]]

# 3. Select columns from Inactive sheet.
#    The Inactive sheet has these column name differences vs Active:
#      'CSG Joining Data'                 -> typo (no 't'), renamed below
#      'Manager Name'                     -> renamed to 'Manager/OM Name'
#      'Lodging - Training start date '   -> trailing space, renamed below
#      'Non-Lodging - Training start date ' -> trailing space, renamed below
hc_inactive = hc_inactive[[
    "Site", "Queue Group", "CSG Joining Data", "Location", "People ID",
    "IEX ID", "OracleID", "Employee Name", "Alias", "Gender",
    "Contract Start Date", "Contract End Date", "Designation", "Grade", "LOB",
    "Role", "Multiple Chat Effective Date", "Primary role",
    "Secondary role", "TL ID", "Supervisor Name", "Supervisor Email",
    "Manager Name",
    "Email Id", "Wave", "CCT Training",
    "Lodging - Training start date ",      # exact name with trailing space
    "Lodging - Training end date",
    "Lodging - Nesting Date", "Lodging - Certification Date (Original)",
    "Lodging - Certification Date (Actual)", "Lodging - Certification status",
    "Non-Lodging - Training start date ",   # exact name with trailing space
    "Non-Lodging - Training end date",
    "Non-Lodging - Nesting Date", "Non-Lodging - Certification Date",
    "Non-Lodging - Certification Date (Actual)", "Non-Lodging - Certification status",
    "Production Date", "AON", "Tenure", "Nationality", "MSA",
    "Current Console Queue", "LWD/Movement", "Attrition Type",
    "Reason for Attrition", "Month", "Week Ending",
]].rename(columns={
    "CSG Joining Data":                    "CSG Joining Date",
    "Manager Name":                        "Manager/OM Name",
    "Lodging - Training start date ":      "Lodging - Training start date",
    "Non-Lodging - Training start date ":  "Non-Lodging - Training start date",
})

# 4. Combine Active and Inactive into a single workforce table
hc_combined = pd.concat([hc_active, hc_inactive], ignore_index=True)

# 5. Build the date spine: 2023-12-01 to 45 days from today
start_date = datetime(2023, 12, 1)
end_date   = datetime.now() + timedelta(days=45)
date_list  = pd.date_range(start_date, end_date)

# 6. Cross-join employees x dates using Polars for speed, then convert to pandas.
#    The original nested list-comprehension was O(n*d) and very memory-heavy.
oracle_ids = hc_combined["OracleID"].dropna().unique().tolist()
pl_dates   = pl.Series("Date", date_list.to_pydatetime())
pl_ids     = pl.Series("OracleID", oracle_ids)

expanded_hc = (
    pl.DataFrame({"OracleID": pl_ids})
      .join(pl.DataFrame({"Date": pl_dates}), how="cross")
      .to_pandas()
)

# 7. Attach employee attributes to each (Date, OracleID) pair
HC_Data_Edition = expanded_hc.merge(hc_combined, on="OracleID", how="left")

# 8. Date columns to convert
date_columns = [
    "Contract Start Date", "Contract End Date", "Multiple Chat Effective Date",
    "CCT Training", "Lodging - Training start date", "Lodging - Training end date",
    "Lodging - Nesting Date", "Lodging - Certification Date (Original)",
    "Lodging - Certification Date (Actual)", "Non-Lodging - Training start date",
    "Non-Lodging - Training end date", "Non-Lodging - Nesting Date",
    "Non-Lodging - Certification Date", "Non-Lodging - Certification Date (Actual)",
    "Production Date", "LWD/Movement", "CSG Joining Date",
]

# 9. Normalize all date columns
#    FIX: original code had the assignment on the same line as the for statement
for column in date_columns:
    HC_Data_Edition[column] = pd.to_datetime(HC_Data_Edition[column], errors="coerce")

# 10. Remove nulls and duplicates
HC_Data_Edition.fillna(pd.NA, inplace=True)
HC_Data_Edition = HC_Data_Edition.sort_values(["OracleID", "Date"])
HC_Data_Edition = HC_Data_Edition.drop_duplicates(subset=["Date", "OracleID"])
HC_Data_Edition = HC_Data_Edition[HC_Data_Edition["OracleID"].notna()]

# 11. Verify columns
print("Column check:", HC_Data_Edition.columns.tolist(), "Done")

# 12. Debug: rows with a matching OracleID but missing Employee Name
print("Rows with missing Employee Name:")
print(
    HC_Data_Edition[HC_Data_Edition["Employee Name"].isna()][
        ["Date", "OracleID", "IEX ID", "Employee Name"]
    ].head(50)
)
print("Done")

print("HC_Data_Edition ready")
HC_Data_Edition


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Cell T576 is marked as a date but the serial value 103145248 is outside the limits for dates. The cell will be treated as an error.
  for idx, row in parser.parse():
c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Cell T716 is marked as a date but the serial value 103145248 is outside the limits for dates. The cell will be treated as an error.
  for idx, row in parser.parse():
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_6312\92473940.py:107: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  HC_Data_Edition[column] = pd.to_datetime(HC_Data_Edition[column], errors="coerce")
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_6312\92473940.py:107: UserWarning: 

Column check: ['OracleID', 'Date', 'Site', 'Queue Group', 'CSG Joining Date', 'Location', 'People ID', 'IEX ID', 'Employee Name', 'Alias', 'Gender', 'Contract Start Date', 'Contract End Date', 'Designation', 'Grade', 'LOB', 'Role', 'Multiple Chat Effective Date', 'Primary role', 'Secondary role', 'TL ID', 'Supervisor Name', 'Supervisor Email', 'Manager/OM Name', 'Email Id', 'Wave', 'CCT Training', 'Lodging - Training start date', 'Lodging - Training end date', 'Lodging - Nesting Date', 'Lodging - Certification Date (Original)', 'Lodging - Certification Date (Actual)', 'Lodging - Certification status', 'Non-Lodging - Training start date', 'Non-Lodging - Training end date', 'Non-Lodging - Nesting Date', 'Non-Lodging - Certification Date', 'Non-Lodging - Certification Date (Actual)', 'Non-Lodging - Certification status', 'Production Date', 'AON', 'Tenure', 'Nationality', 'MSA', 'Current Console Queue', 'LWD/Movement', 'Attrition Type', 'Reason for Attrition', 'Month', 'Week Ending'] Done


,OracleID,Date,Site,Queue Group,CSG Joining Date,Location,People ID,IEX ID,Employee Name,Alias,...,AON,Tenure,Nationality,MSA,Current Console Queue,LWD/Movement,Attrition Type,Reason for Attrition,Month,Week Ending
230790,1081503,2023-12-01 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
230791,1081503,2023-12-02 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
230792,1081503,2023-12-03 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
230793,1081503,2023-12-04 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
230794,1081503,2023-12-05 00:00:00,-,-,NaT,Vietnam,178879477,<NA>,TRAN PHAM NGOC VAN,Van,...,-,-,Vietnamese,Expedia,-,2024-05-02,Movement,Movement to Etraveli,May'24,2024-05-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131875,103579593,2026-06-25 00:00:00,ONEHUB,-,NaT,Vietnam,560932699,3014279,DOAN QUOC THUAN,Mark,...,-,-,Vietnamese,Expedia,<NA>,NaT,<NA>,<NA>,<NA>,NaT
131876,103579593,2026-06-26 00:00:00,ONEHUB,-,NaT,Vietnam,560932699,3014279,DOAN QUOC THUAN,Mark,...,-,-,Vietnamese,Expedia,<NA>,NaT,<NA>,<NA>,<NA>,NaT
131877,103579593,2026-06-27 00:00:00,ONEHUB,-,NaT,Vietnam,560932699,3014279,DOAN QUOC THUAN,Mark,...,-,-,Vietnamese,Expedia,<NA>,NaT,<NA>,<NA>,<NA>,NaT
131878,103579593,2026-06-28 00:00:00,ONEHUB,-,NaT,Vietnam,560932699,3014279,DOAN QUOC THUAN,Mark,...,-,-,Vietnamese,Expedia,<NA>,NaT,<NA>,<NA>,<NA>,NaT


In [110]:
# ---------------------------------------------------------------------------
# Map each (Date, OracleID) snapshot to the most recent Effective Date entry
# in Staffing_Records that does not exceed the snapshot date.
# This implements a "last-value-carried-forward" (LOCF) join to reconstruct
# the correct HR status for any given day.
# ---------------------------------------------------------------------------

# 1. Normalise date types
HC_Data_Edition["Date"]           = pd.to_datetime(HC_Data_Edition["Date"],            errors="coerce")
staffing_records["Effective Date"] = pd.to_datetime(staffing_records["Effective Date"], errors="coerce")

# 2. Extract unique (Date, OracleID) pairs from the expanded headcount table
HC_Data_Edition_Only_ID   = HC_Data_Edition[["Date", "OracleID"]]
staffing_records_date      = staffing_records[["Effective Date", "OracleID"]].rename(
    columns={"OracleID": "Changed.OracleID"}
)

# 3. Keep only snapshots whose OracleID exists in Staffing_Records
merged_id = pd.merge(
    HC_Data_Edition_Only_ID,
    staffing_records_date[["Changed.OracleID"]],
    how="left", left_on="OracleID", right_on="Changed.OracleID",
)
merged_id = merged_id[merged_id["Changed.OracleID"].notna()].drop(columns=["Changed.OracleID"])

# 4. Left-join on (Date, OracleID) ↔ (Effective Date, Changed.OracleID)
#    to identify snapshots that coincide with a staffing change
merged_effective_date = pd.merge(
    merged_id,
    staffing_records_date[["Effective Date", "Changed.OracleID"]],
    how="left",
    left_on=["Date", "OracleID"],
    right_on=["Effective Date", "Changed.OracleID"],
)
merged_effective_date = (
    merged_effective_date
    .sort_values(["OracleID", "Date"])
    .drop_duplicates(subset=["Date", "OracleID", "Effective Date", "Changed.OracleID"])
)

# 5. Forward-fill Effective Date so every snapshot row carries the last known status
merged_effective_date["Effective Date"] = merged_effective_date["Effective Date"].ffill()

# 6. Enforce business rule: Effective Date must not exceed the snapshot Date
mask = merged_effective_date["Effective Date"] > merged_effective_date["Date"]
merged_effective_date.loc[mask, "Effective Date"] = pd.NaT
merged_effective_date = merged_effective_date[merged_effective_date["Effective Date"].notna()]
merged_effective_date = merged_effective_date.drop(columns=["Changed.OracleID"])

print("Columns after Effective Date mapping:\n", merged_effective_date.columns.tolist(), "\nDone\n")

# 7. Attach full staffing record attributes for each (Effective Date, OracleID)
merged_staffing_records = pd.merge(
    merged_effective_date, staffing_records,
    how="left", on=["Effective Date", "OracleID"],
)

# 8. Pull attrition columns from the expanded headcount table
HC_Data_Edition_Inactive = HC_Data_Edition[[
    "Date", "Site", "Queue Group", "CSG Joining Date",
    "OracleID", "LWD/Movement", "Attrition Type", "Reason for Attrition",
]]
merged_inactive_cols = pd.merge(
    merged_staffing_records, HC_Data_Edition_Inactive,
    on=["Date", "OracleID"], how="left",
)
staffing_records = merged_inactive_cols

# 9. Identify snapshot rows where no new Effective Date was recorded
#    (i.e. no staffing change on that day → carry-over from previous state)
merged_staffing_records = HC_Data_Edition.merge(
    staffing_records[["Date", "OracleID", "Effective Date"]],
    how="left", on=["Date", "OracleID"],
)
HC_Data_Edition_not_change = merged_staffing_records[merged_staffing_records["Effective Date"].isna()]

# 10. Combine changed and unchanged rows into the final expanded table
hc_extend = pd.concat([HC_Data_Edition_not_change, staffing_records], ignore_index=True)
hc_extend = hc_extend.drop(columns=["Effective Date"])


Columns after Effective Date mapping:
 ['Date', 'OracleID', 'Effective Date'] 
Done



In [ ]:
# ---------------------------------------------------------------------------
# Normalise column names, compute production milestones, Detail Status,
# AON (Age on Nesting/Production) and tenure buckets (Lodging & Non-Lodging).
#
# BUG FIXES:
#   1. get_next_monday / get_aon / get_lg_tenure / get_nl_tenure used slow
#      row-by-row apply; replaced with vectorised numpy / pandas operations.
#   2. np.where returns an object array when passed pd.NaT as the 'true' value;
#      result is cast to datetime64 explicitly after the np.where call.
# ---------------------------------------------------------------------------

# 1. Drop redundant end-date columns (already implicit in Nesting / Production dates)
hc_extend = hc_extend.drop(
    columns=["Lodging - Training end date", "Non-Lodging - Training end date"],
    errors="ignore",
)

# 2. Rename columns to short, readable names used throughout the rest of the notebook
hc_extend = hc_extend.rename(columns={
    "Lodging - Training start date":          "Lodging Training",
    "Lodging - Nesting Date":                 "Lodging Nesting",
    "Lodging - Certification Date (Original)":"Lodging Original",
    "Lodging - Certification Date (Actual)":  "Lodging Actual",
    "Lodging - Certification status":         "Lodging",
    "Non-Lodging - Training start date":      "NonLodging Training",
    "Non-Lodging - Nesting Date":             "NonLodging Nesting",
    "Non-Lodging - Certification Date":       "NonLodging Original",
    "Non-Lodging - Certification Date (Actual)": "NonLodging Actual",
    "Non-Lodging - Certification status":     "NonLodging",
    "Production Date":                        "Production",
})

# 3. Parse all date columns to datetime64
date_columns = [
    "Date", "CCT Training", "Lodging Training", "Lodging Nesting",
    "Lodging Original", "Lodging Actual", "NonLodging Training",
    "NonLodging Nesting", "NonLodging Actual", "Production",
    "NonLodging Original", "Multiple Chat Effective Date",
    "LWD/Movement", "CSG Joining Date",
]
for col in date_columns:
    if col in hc_extend.columns:
        hc_extend[col] = pd.to_datetime(hc_extend[col], errors="coerce")

# 4. Vectorised "next Monday" helper — returns the Monday on or after a date.
#    offset_days = (7 - weekday) % 7  → 0 for Monday, 6 for Tuesday, …, 1 for Sunday
def next_monday_series(s: pd.Series) -> pd.Series:
    s = pd.to_datetime(s, errors="coerce")
    offsets = (7 - s.dt.weekday) % 7      # Monday = 0 → offset 0
    return s + pd.to_timedelta(offsets, unit="D")

# 5. Production Date
hc_extend["Lodging Production"]    = pd.to_datetime(hc_extend["Lodging Actual"],    errors="coerce")
hc_extend["NonLodging Production"] = pd.to_datetime(hc_extend["NonLodging Actual"], errors="coerce")

# 6. Extended Nesting
lg_has_extend  = hc_extend["Lodging Original"].notna()  & hc_extend["Lodging Actual"].notna()  & (hc_extend["Lodging Original"]  < hc_extend["Lodging Actual"])
nl_has_extend  = hc_extend["NonLodging Original"].notna() & hc_extend["NonLodging Actual"].notna() & (hc_extend["NonLodging Original"] < hc_extend["NonLodging Actual"])

hc_extend["Lodging Extended Nesting"]    = hc_extend["Lodging Original"].where(lg_has_extend)
hc_extend["NonLodging Extended Nesting"] = hc_extend["NonLodging Original"].where(nl_has_extend)

# 7. Drop helper / intermediate columns no longer needed
cols_to_remove = [
    "Next Monday Lodging", "Next Monday NonLodging", "Lodging Days", "NonLodging Days",
    "Lodging Original", "NonLodging Original", "Lodging Actual", "Lodging",
    "NonLodging Actual", "NonLodging", "Contract Start Date", "Contract End Date",
]
hc_extend = hc_extend.drop(columns=[c for c in cols_to_remove if c in hc_extend.columns])

# 8. Reorder columns to the canonical dashboard layout
column_order = [
    "Date", "Site", "Queue Group", "OracleID", "People ID", "IEX ID",
    "Employee Name", "Alias", "Gender", "Designation", "Grade", "LOB",
    "Multiple Chat Effective Date", "Role", "Primary role", "Secondary role",
    "TL ID", "Supervisor Name", "Supervisor Email", "Manager/OM Name", "Email Id",
    "Wave", "CCT Training", "Lodging Training", "Lodging Nesting",
    "Lodging Extended Nesting", "Lodging Production", "NonLodging Training",
    "NonLodging Nesting", "NonLodging Extended Nesting", "NonLodging Production",
    "Production", "Attrition Type", "Reason for Attrition", "ATT_Reason_LV3",
    "LWD/Movement", "CSG Joining Date", "Current Console Queue",
]
hc_extend = hc_extend[[c for c in column_order if c in hc_extend.columns]]
hc_extend = hc_extend.sort_values("Date")

# 9. Re-confirm date dtypes after column reorder
date_cols_recheck = [
    "Date", "NonLodging Production", "NonLodging Extended Nesting",
    "NonLodging Nesting", "NonLodging Training", "Lodging Production",
    "Lodging Extended Nesting", "Lodging Nesting", "Lodging Training", "CCT Training",
]
for col in date_cols_recheck:
    if col in hc_extend.columns:
        hc_extend[col] = pd.to_datetime(hc_extend[col], errors="coerce")

# 10. Vectorised Detail Status computation.
#     Priority: Non-Lodging Production > Extended Nesting > Nesting > Training >
#               Lodging Production > Extended Nesting > Nesting > Training >
#               CCT Training > Unavailable
d     = hc_extend["Date"]
nl_p  = hc_extend["NonLodging Production"]
nl_ex = hc_extend["NonLodging Extended Nesting"]
nl_n  = hc_extend["NonLodging Nesting"]
nl_t  = hc_extend["NonLodging Training"]
l_p   = hc_extend["Lodging Production"]
l_ex  = hc_extend["Lodging Extended Nesting"]
l_n   = hc_extend["Lodging Nesting"]
l_t   = hc_extend["Lodging Training"]
cct   = hc_extend["CCT Training"]

hc_extend["Detail Status"] = np.select(
    [
        d.notna() & nl_p.notna()  & (d >= nl_p),
        d.notna() & nl_ex.notna() & nl_p.isna()   & (d >= nl_ex),
        d.notna() & nl_ex.notna() & nl_p.notna()  & (nl_ex <= d) & (d < nl_p),
        d.notna() & nl_n.notna()  & nl_ex.isna()  & (d >= nl_n),
        d.notna() & nl_n.notna()  & nl_ex.notna() & (nl_n <= d) & (d < nl_ex),
        d.notna() & nl_t.notna()  & nl_n.isna()   & (d >= nl_t),
        d.notna() & nl_t.notna()  & nl_n.notna()  & (nl_t <= d) & (d < nl_n),
        d.notna() & l_p.notna()   & nl_t.isna()   & (d >= l_p),
        d.notna() & l_p.notna()   & nl_t.notna()  & (l_p <= d) & (d < nl_t),
        d.notna() & l_ex.notna()  & l_p.isna()    & (d >= l_ex),
        d.notna() & l_ex.notna()  & l_p.notna()   & (l_ex <= d) & (d < l_p),
        d.notna() & l_n.notna()   & l_ex.isna()   & (d >= l_n),
        d.notna() & l_n.notna()   & l_ex.notna()  & (l_n <= d) & (d < l_ex),
        d.notna() & l_t.notna()   & l_n.isna()    & (d >= l_t),
        d.notna() & l_t.notna()   & l_n.notna()   & (l_t <= d) & (d < l_n),
        d.notna() & cct.notna()   & l_t.isna()    & (d >= cct),
        d.notna() & cct.notna()   & l_t.notna()   & (cct <= d) & (d < l_t),
        d.notna() & cct.notna()   & (d < cct),
    ],
    [
        "Non Lodging Production", "Non Lodging Extended Nesting", "Non Lodging Extended Nesting",
        "Non Lodging Nesting",    "Non Lodging Nesting",
        "Non Lodging Training",   "Non Lodging Training",
        "Lodging Production",     "Lodging Production",
        "Lodging Extended Nesting", "Lodging Extended Nesting",
        "Lodging Nesting",        "Lodging Nesting",
        "Lodging Training",       "Lodging Training",
        "CCT Training",           "CCT Training",
        "Unavailable",
    ],
    default="-",
)

# 11. Vectorised AON (Age on Nesting/Production) in days
hc_extend["AON"] = np.where(
    hc_extend["Detail Status"] == "Lodging Production",
    (d - l_p).dt.days,
    np.where(
        hc_extend["Detail Status"] == "Non Lodging Production",
        (d - nl_p).dt.days,
        np.nan,
    )
)
hc_extend["AON"] = pd.to_numeric(hc_extend["AON"], errors="coerce")

# 12. Vectorised LG Tenure label
aon = hc_extend["AON"]
lg_status  = hc_extend["Detail Status"]
hc_extend["LG Tenure"] = np.select(
    [
        lg_status.isin(["Lodging Nesting", "Lodging Extended Nesting"]),
        (l_p.notna()) & (d >= l_p) & (aon.between(0, 30)),
        (l_p.notna()) & (d >= l_p) & (aon.between(31, 60)),
        (l_p.notna()) & (d >= l_p) & (aon.between(61, 90)),
        (l_p.notna()) & (d >= l_p) & (aon > 90),
    ],
    ["Nesting", "0-30 Days", "30-60 Days", "60-90 Days", "> 90 Days"],
    default=None,
)

# 13. Vectorised NL Tenure label
hc_extend["NL Tenure"] = np.select(
    [
        lg_status.isin(["Non Lodging Nesting", "Non Lodging Extended Nesting"]),
        (nl_p.notna()) & (d >= nl_p) & (aon.between(0, 30)),
        (nl_p.notna()) & (d >= nl_p) & (aon.between(31, 60)),
        (nl_p.notna()) & (d >= nl_p) & (aon.between(61, 90)),
        (nl_p.notna()) & (d >= nl_p) & (aon > 90),
    ],
    ["Nesting", "0-30 Days", "30-60 Days", "60-90 Days", "> 90 Days"],
    default=None,
)

print(hc_extend.columns.tolist())


In [ ]:
# ---------------------------------------------------------------------------
# Enrich the daily headcount snapshot with derived analytical columns:
#   Status, Concurrency, LOB_2, LOB_3, Performance_Calculation, Stage,
#   ATT_Reason_LV3, HC Open/Close by Week/Month, ATT, Movement, CSG, Active.
#
# BUG FIX: HC Open/Close and ATT/Movement were computed twice in the original
#   cell (steps 12-13 and then again verbatim at the bottom), causing silent
#   overwrites. The duplicate block has been removed.
# ---------------------------------------------------------------------------

# 1. Status: Terminated / Transferred if past LWD, else Active if training started
hc_extend["Status"] = np.where(
    hc_extend["LWD/Movement"].notna() & (hc_extend["Date"] >= hc_extend["LWD/Movement"]),
    np.where(hc_extend["Attrition Type"] == "Movement", "Transfered", "Terminated"),
    np.where(hc_extend["Date"] >= hc_extend["CCT Training"], "Active", None),
).astype(object)

# 2. Concurrency: 2 if Multiple Chat is active and status is Production/Nesting; else 1 or 0
hc_extend["Concurrency"] = np.where(
    hc_extend["Detail Status"].str.contains("Production|Nesting", na=False),
    np.where(
        hc_extend["Multiple Chat Effective Date"].notna()
        & (hc_extend["Date"] >= hc_extend["Multiple Chat Effective Date"]),
        2, 1,
    ),
    0,
)

# 3. LOB_2: classify each row's Line-of-Business by status and channel (Chat/Voice)
d_status = hc_extend["Detail Status"]
lob      = hc_extend["LOB"]
role     = hc_extend["Role"]
hc_extend["LOB_2"] = np.select(
    [
        d_status.str.contains("Production", na=False) & (lob == "Lodging")     & role.str.contains("Chat",  na=False),
        d_status.str.contains("Production", na=False) & (lob == "Non_Lodging") & role.str.contains("Chat",  na=False),
        d_status.isin(["Lodging Extended Nesting",     "Lodging Nesting"])      & role.str.contains("Chat",  na=False),
        d_status.isin(["Non Lodging Extended Nesting", "Non Lodging Nesting"])  & role.str.contains("Chat",  na=False),
        d_status.str.contains("Production", na=False) & (lob == "Lodging")     & role.str.contains("Voice", na=False),
        d_status.str.contains("Production", na=False) & (lob == "Non_Lodging") & role.str.contains("Voice", na=False),
        d_status.isin(["Lodging Extended Nesting",     "Lodging Nesting"])      & role.str.contains("Voice", na=False),
        d_status.isin(["Non Lodging Extended Nesting", "Non Lodging Nesting"])  & role.str.contains("Voice", na=False),
    ],
    [
        "Lodging Chat", "Non Lodging Chat",
        "Lodging Nesting Chat", "Non Lodging Nesting Chat",
        "Lodging Voice", "Non Lodging Voice",
        "Lodging Nesting Voice", "Non Lodging Nesting Voice",
    ],
    default=None,
).astype(object)

hc_extend["LOB_3"] = hc_extend["LOB_2"]

# 4. Sanitise fields that may carry pandas NA objects
hc_extend[["Detail Status", "AON", "LG Tenure", "NL Tenure", "Concurrency", "Status"]] = (
    hc_extend[["Detail Status", "AON", "LG Tenure", "NL Tenure", "Concurrency", "Status"]]
    .replace({pd.NA: np.nan})
)

# 5. Performance_Calculation: exclude rows still in training or unavailable
hc_extend["Performance_Calculation"] = np.where(
    hc_extend["Detail Status"].isna()
    | hc_extend["Detail Status"].isin(["CCT Training", "Lodging Training", "Non Lodging Training", "Unavailable"]),
    "Excluded", "Included",
)

# 6. Remove rows that haven't entered any pipeline yet
hc_extend = hc_extend[hc_extend["Detail Status"] != "Unavailable"]

# 7. Deduplicate on (Date, IEX ID, Employee Name) — each snapshot must be unique
hc_extend = hc_extend.drop_duplicates(subset=["Date", "IEX ID", "Employee Name"])

# 8. Enrich with calendar metadata from the Date_Format sheet
hc_extend = pd.merge(hc_extend, date_format, on="Date", how="left")

# 9. Stage: high-level grouping of Detail Status
hc_extend["Stage"] = np.select(
    [
        hc_extend["Detail Status"].str.contains("Production", na=False),
        hc_extend["Detail Status"].str.contains("Training",   na=False),
        hc_extend["Detail Status"].str.contains("Nesting",    na=False),
    ],
    ["In Production", "In Training", "In Nesting"],
    default=None,
)

# 10. ATT_Reason_LV3: extract the reason text after the first 4 space-delimited tokens
hc_extend["ATT_Reason_LV3"] = (
    hc_extend["Reason for Attrition"]
    .str.split(" ")
    .str[4:]
    .str.join(" ")
)

# 11. HC Open / Closed counters (1 on the boundary date, 0 elsewhere)
lwd_mask = hc_extend["LWD/Movement"].isna() | (hc_extend["Date"] <= hc_extend["LWD/Movement"])

hc_extend["HC Open By Week"]   = np.where(lwd_mask & (hc_extend["Date"] == hc_extend["Date Start Week"]),  1, 0)
hc_extend["HC Closed By Week"] = np.where(lwd_mask & (hc_extend["Date"] == hc_extend["Date End Week"]),    1, 0)
hc_extend["HC Open By Month"]  = np.where(lwd_mask & (hc_extend["Date"] == hc_extend["Date Start Month"]), 1, 0)
hc_extend["HC Closed By Month"]= np.where(lwd_mask & (hc_extend["Date"] == hc_extend["Date End Month"]),   1, 0)

# 12. ATT / Movement flags: fired on the day after LWD
hc_extend["ATT/Movement"] = np.where(
    hc_extend["Date"] == hc_extend["LWD/Movement"] + pd.Timedelta(days=1), 1, 0
)
hc_extend["ATT"] = np.where(
    (hc_extend["ATT/Movement"] == 1)
    & hc_extend["Attrition Type"].isin(["Voluntary", "Involuntary"]),
    1, 0,
)
hc_extend["Movement"] = hc_extend["ATT/Movement"] - hc_extend["ATT"]

# 13. Fill any remaining NaN in numeric event flags with 0
for col in ["ATT/Movement", "ATT", "Movement"]:
    hc_extend[col] = hc_extend[col].fillna(0)

# 14. Active flag and CSG enrolment flag
hc_extend["Active"] = np.where(hc_extend["LWD/Movement"].isna(), "Yes", "No")
hc_extend["CSG"]    = np.where(
    hc_extend["CSG Joining Date"].isna() | (hc_extend["Date"] < hc_extend["CSG Joining Date"]),
    0, 1,
)

# 15. Final column order for Power BI / Tableau / CSV export
column_order = [
    "Year", "Month", "Site", "Queue Group", "Week Begin",
    "Date Start Month", "Date End Month", "Date Start Week", "Date End Week",
    "Week", "Day", "Date", "OracleID", "People ID", "IEX ID", "Employee Name",
    "Alias", "Gender", "Designation", "Grade", "LOB", "Role",
    "Multiple Chat Effective Date", "Primary role", "Secondary role",
    "TL ID", "Supervisor Name", "Supervisor Email", "Manager/OM Name",
    "Email Id", "Wave", "CCT Training", "Lodging Training", "Lodging Nesting",
    "Lodging Extended Nesting", "CSG Joining Date", "Lodging Production",
    "NonLodging Training", "NonLodging Nesting", "NonLodging Extended Nesting",
    "NonLodging Production", "Production", "LWD/Movement",
    "Reason for Attrition", "Attrition Type", "ATT_Reason_LV3",
    "Detail Status", "AON", "LG Tenure", "NL Tenure", "Concurrency",
    "Status", "LOB_2", "LOB_3", "Performance_Calculation", "Stage",
    "HC Open By Week", "HC Closed By Week", "HC Open By Month", "HC Closed By Month",
    "ATT/Movement", "ATT", "Movement", "CSG", "Active", "Current Console Queue",
]
hc_extend = hc_extend[[c for c in column_order if c in hc_extend.columns]]
hc_extend.reset_index(drop=True, inplace=True)


In [ ]:
hc_extend['Date']


0        2023-12-01
1        2023-12-01
2        2023-12-01
3        2023-12-01
4        2023-12-01
            ...    
559767   2026-06-26
559768   2026-06-26
559769   2026-06-26
559770   2026-06-26
559771   2026-06-26
Name: Date, Length: 559772, dtype: datetime64[ns]

In [ ]:
# ---------------------------------------------------------------------------
# Map Mini Team / Mini TL information onto each daily snapshot.
# The mini_team table can have multiple effective-date entries per employee,
# so we pick the most recent one that does not exceed the snapshot date.
# ---------------------------------------------------------------------------

# 1. Read the Mini Team sheet and select the relevant columns
mini_team = pd.read_excel(folder_paths["hc_staffing"], sheet_name="Mini Team")

mini_team = mini_team[[
    "Mini TL", "Mini TL - Email", "Emp Email",
    "Role", "Active", "Mini TL - Short Name", "Effective Date",
]].rename(columns={"Emp Email": "Email Id"})

# 2. Normalise date columns
mini_team["Effective Date"]  = pd.to_datetime(mini_team["Effective Date"], errors="coerce")
hc_extend["Date"]            = pd.to_datetime(hc_extend["Date"],           errors="coerce")

# 3. Unique (Date, Email Id) pairs from the expanded headcount table
hc_extend_email_date = hc_extend[["Date", "Email Id"]].drop_duplicates()

# 4. Left-join to get all effective dates for each email
merge_key = pd.merge(
    hc_extend_email_date,
    mini_team[["Email Id", "Effective Date"]],
    on="Email Id", how="left",
)

# 5. Keep only effective dates that are on or before the snapshot date
merge_key = merge_key[merge_key["Effective Date"] <= merge_key["Date"]]

# 6. Per (Email Id, Date): retain only the latest effective date
merge_key = (
    merge_key
    .sort_values(["Email Id", "Date", "Effective Date"])
    .drop_duplicates(subset=["Email Id", "Date"], keep="last")
)

# 7. Attach the Mini TL details for that effective date
merge_mini_team = pd.merge(
    merge_key,
    mini_team[["Email Id", "Effective Date", "Mini TL - Email", "Mini TL - Short Name"]],
    on=["Email Id", "Effective Date"], how="left",
).sort_values("Date")

# 8. Join Mini TL info back onto the main headcount table
hc_extend = pd.merge(
    hc_extend,
    merge_mini_team[["Email Id", "Date", "Mini TL - Email", "Mini TL - Short Name", "Effective Date"]],
    on=["Email Id", "Date"], how="left",
)

# 9. Rename 'Effective Date' to 'Mini TL Start Date' for clarity
hc_extend = hc_extend.rename(columns={"Effective Date": "Mini TL Start Date"})

hc_extend


,Year,Month,Site,Queue Group,Week Begin,Date Start Month,Date End Month,Date Start Week,Date End Week,Week,...,HC Closed By Month,ATT/Movement,ATT,Movement,CSG,Active,Current Console Queue,Mini TL - Email,Mini TL - Short Name,Mini TL Start Date
0,2023,Dec-23,-,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,0,0,0,No,-,NaN,NaN,NaT
1,2023,Dec-23,-,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,0,0,0,No,-,NaN,NaN,NaT
2,2023,Dec-23,ONEHUB,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,0,0,0,No,-,NaN,NaN,NaT
3,2023,Dec-23,FLEMINGTON,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,0,0,0,No,-,NaN,NaN,NaT
4,2023,Dec-23,FLEMINGTON,-,WB2711,2023-12-01,2023-12-31,2023-11-27,2023-12-03,49,...,0,0,0,0,0,No,-,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559827,2026,Jun-26,ONEHUB,-,WB2206,2026-06-01,2026-06-30,2026-06-22,2026-06-28,26,...,0,0,0,0,0,No,Chat_OD_EN_Dual_GDS,NaN,NaN,NaT
559828,2026,Jun-26,ONEHUB,-,WB2206,2026-06-01,2026-06-30,2026-06-22,2026-06-28,26,...,0,0,0,0,0,No,Chat_OD_EN_Lodging,NaN,NaN,NaT
559829,2026,Jun-26,ONEHUB,CSG,WB2206,2026-06-01,2026-06-30,2026-06-22,2026-06-28,26,...,0,0,0,0,1,Yes,Chat_OD_EN_Lodging,thienkim.chau@concentrix.com,Kim,2025-08-01
559830,2026,Jun-26,ONEHUB,-,WB2206,2026-06-01,2026-06-30,2026-06-22,2026-06-28,26,...,0,0,0,0,0,No,Chat_OD_EN_Lodging,NaN,NaN,NaT


In [ ]:
# ---------------------------------------------------------------------------
# Enforce the final schema: cast every column to its target dtype so that
# Power BI / Tableau imports are clean and memory usage is minimised.
# ---------------------------------------------------------------------------

dtype_map = {
    "Year": "Int64", "Month": "string", "Site": "string", "Queue Group": "string",
    "Week Begin": "string", "Week": "Int64", "Day": "string",
    "OracleID": "Int64", "People ID": "Int64", "IEX ID": "Int64",
    "Employee Name": "string", "Alias": "string", "Gender": "string",
    "Designation": "string", "Grade": "string", "LOB": "string",
    "Role": "string", "Primary role": "string", "Secondary role": "string",
    "TL ID": "Int64", "Supervisor Name": "string", "Supervisor Email": "string",
    "Manager/OM Name": "string", "Email Id": "string",
    "Reason for Attrition": "string", "Attrition Type": "string",
    "ATT_Reason_LV3": "string", "Detail Status": "string",
    "LG Tenure": "string", "NL Tenure": "string", "Status": "string",
    "LOB_2": "string", "LOB_3": "string",
    "Performance_Calculation": "string", "Stage": "string",
    "Wave": "object",
    "AON": "Int64", "Concurrency": "Int64",
    "HC Open By Week": "Int64", "HC Closed By Week": "Int64",
    "HC Open By Month": "Int64", "HC Closed By Month": "Int64",
    "ATT/Movement": "Int64", "ATT": "Int64", "Movement": "Int64",
    "CSG": "Int64", "Active": "string",
    "Mini TL - Email": "string", "Mini TL - Short Name": "string",
    "Current Console Queue": "string",
}

# Columns that should be stored as plain date (no time component)
date_cols = [
    "Date Start Month", "Date End Month", "Date Start Week", "Date End Week",
    "Date", "Multiple Chat Effective Date", "CCT Training",
    "Lodging Training", "Lodging Nesting", "Lodging Extended Nesting",
    "CSG Joining Date", "Lodging Production", "NonLodging Training",
    "NonLodging Nesting", "NonLodging Extended Nesting", "NonLodging Production",
    "Production", "LWD/Movement", "Mini TL Start Date",
]

# Convert date columns to Python date objects (strip time component)
for col in date_cols:
    if col in hc_extend.columns:
        hc_extend[col] = pd.to_datetime(hc_extend[col], errors="coerce").dt.date

# Apply dtype_map to all remaining columns
for col, dtype in dtype_map.items():
    if col not in hc_extend.columns:
        continue
    if dtype == "Int64":
        hc_extend[col] = pd.to_numeric(hc_extend[col], errors="coerce").astype("Int64")
    elif dtype == "string":
        hc_extend[col] = hc_extend[col].astype("string")
    else:
        hc_extend[col] = hc_extend[col].astype(dtype)

print(hc_extend.dtypes)


Year                              Int64
Month                    string[python]
Site                     string[python]
Queue Group              string[python]
Week Begin               string[python]
                              ...      
Active                   string[python]
Current Console Queue    string[python]
Mini TL - Email          string[python]
Mini TL - Short Name     string[python]
Mini TL Start Date               object
Length: 69, dtype: object


In [ ]:
# ---------------------------------------------------------------------------
# Build a weekly HC Role pivot for 2026 (Mondays only).
# Each row = one employee; each column = one week (Date Start Week value);
# cell value = the HC_File_Role label for that week.
#
# EWS INTEGRATION:
#   Employees with an Expected Last Working Day (LWD Expected) in the EWS
#   sheet are marked as 'LG Chat Termed' or 'NL Chat Termed' on any pivot
#   week that falls ON or AFTER their LWD Expected date. This allows the
#   pivot to be used directly for 1-month headcount forecasting.
# ---------------------------------------------------------------------------

# 1. Designation groups
desig_advisor = ["Advisor I, Customer Service"]
desig_sme     = ["SME, Operations", "Sr. SME, Operations"]
desig_tl_ops  = ["Supervisor, Business Operations", "Team Leader, Operations"]
desig_qa      = ["Sr. Quality Evaluator", "Quality Evaluator"]
desig_rta     = ["Sr. Representative, Real Time Management", "Associate, Real Time Management"]
desig_tl_wfm  = ["Supervisor, WFM"]
desig_trainer  = ["Communications Trainer II", "Communications Trainer I", "Trainer I"]
desig_leader   = [
    "Sr. Supervisor, WFM", "Supervisor, WFM", "Sr. Service Delivery Manager",
    "Supervisor, Training & Quality", "Manager II, Training & Quality",
    "Operations Manager I", "Supervisor, Business Operations",
    "Associate, Business Intelligence", "Associate Director, Operations Support",
    "Sr. Quality Evaluator", "Sr. Manager, Training & Quality",
    "Sr. Supervisor, Training & Quality", "Manager I, WFM",
    "Service Delivery Manager II", "Manager I, Communications Training",
    "Analyst, WFM Real Time Management",
]

# 2. Filter to 2026 Mondays only
is_2026   = hc_extend["Year"].isin([2026])
is_monday = pd.to_datetime(hc_extend["Date"], errors="coerce").dt.weekday == 0
hc_monday = hc_extend[is_2026 & is_monday].copy().reset_index(drop=True)

# 3. Pre-compute boolean masks once
stat_active = hc_monday["Status"] == "Active"
stat_termed = hc_monday["Status"] == "Terminated"
stat_trans  = hc_monday["Status"] == "Transfered"

dsg_adv = hc_monday["Designation"].isin(desig_advisor)
dsg_sme = hc_monday["Designation"].isin(desig_sme)
dsg_tlo = hc_monday["Designation"].isin(desig_tl_ops)
dsg_qa  = hc_monday["Designation"].isin(desig_qa)
dsg_rta = hc_monday["Designation"].isin(desig_rta)
dsg_tlw = hc_monday["Designation"].isin(desig_tl_wfm)
dsg_trn = hc_monday["Designation"].isin(desig_trainer)
dsg_ldr = hc_monday["Designation"].isin(desig_leader)

lob_supp_lg  = hc_monday["LOB"].str.contains("(?i)Support_LG",      na=False, regex=True)
lob_supp_nl  = hc_monday["LOB"].str.contains("(?i)Support_NL",      na=False, regex=True)
lob_flex_trn = hc_monday["LOB"].str.contains("(?i)Flex_Trainer",     na=False, regex=True)
lob_flex_qa  = hc_monday["LOB"].str.contains("(?i)Flex_QA",          na=False, regex=True)
lob_flex_sme = hc_monday["LOB"].str.contains("(?i)Flex_SME",         na=False, regex=True)
lob_nl       = hc_monday["LOB"].str.contains("Non_Lodging|NL",        na=False, regex=True)
lob_lg       = hc_monday["LOB"].str.contains("Lodging|LG",            na=False, regex=True)
lob_supp     = hc_monday["LOB"].str.contains("(?i)Support",           na=False, regex=True)

dtl_nl_trn  = hc_monday["Detail Status"] == "Non Lodging Training"
dtl_lg_trn  = hc_monday["Detail Status"] == "Lodging Training"
dtl_nesting = hc_monday["Detail Status"].str.contains("(?i)Nesting", na=False, regex=True)

# 4. Build conditions and choices for np.select
conditions = [
    dsg_adv & lob_supp_lg  & stat_active,
    dsg_adv & lob_supp_nl  & stat_active,
    dsg_adv & lob_flex_trn & stat_active,
    dsg_adv & lob_flex_qa  & stat_active,
    dsg_adv & lob_flex_sme & stat_active,
    dsg_adv & lob_nl & dtl_nl_trn  & stat_active,
    dsg_adv & lob_nl & dtl_nesting & stat_active,
    dsg_adv & lob_nl & stat_termed,
    dsg_adv & lob_nl & stat_trans,
    dsg_adv & lob_nl & stat_active,
    dsg_adv & lob_lg & dtl_lg_trn  & stat_active,
    dsg_adv & lob_lg & dtl_nesting & stat_active,
    dsg_adv & lob_lg & stat_termed,
    dsg_adv & lob_lg & stat_trans,
    dsg_adv & lob_lg & stat_active,
    dsg_sme & lob_nl & stat_active,
    dsg_sme & lob_nl & stat_termed,
    dsg_sme & lob_lg & stat_active,
    dsg_sme & lob_lg & stat_termed,
    dsg_sme & lob_supp & stat_active,
    dsg_sme & lob_supp & stat_termed,
    dsg_tlo & lob_nl & stat_active,
    dsg_tlo & lob_nl & stat_termed,
    dsg_tlo & lob_lg & stat_active,
    dsg_tlo & lob_lg & stat_termed,
    dsg_tlo & lob_supp & stat_active,
    dsg_tlo & lob_supp & stat_termed,
    stat_trans,
    dsg_tlo & stat_termed,
    dsg_qa  & stat_termed,
    dsg_rta & stat_termed,
    dsg_trn & stat_termed,
    dsg_qa  & stat_active,
    dsg_rta & stat_active,
    dsg_tlw & stat_active,
    dsg_ldr & stat_termed,
    dsg_ldr & stat_active,
]
choices = [
    "LG Chat Task", "NL Chat Task", "Trainer Task", "QA Task", "SME Task",
    "NL Chat Training", "NL Chat Nesting", "NL Chat Termed", "NL Chat Transferred", "NL Chat",
    "LG Chat Training", "LG Chat Nesting", "LG Chat Termed", "LG Chat Transferred", "LG Chat",
    "NL Chat SME", "NL Chat SME Termed", "LG Chat SME", "LG Chat SME Termed", "SME", "Termed",
    "NL Chat TL",  "NL Chat TL Termed",  "LG Chat TL",  "LG Chat Termed", "TL",  "Termed",
    "Transferred",
    "Retail TL Termed", "Retail QA Termed", "RTA Termed", "Retail Trainer Termed",
    "Retail QA", "RTA", "TL",
    "Leader Termed", "Leader",
]

conditions_clean = [c.fillna(False).astype(bool) for c in conditions]
hc_monday["HC_File_Role"] = np.select(conditions_clean, choices, default=None)

# ---------------------------------------------------------------------------
# 5. EWS integration
#    Read the EWS sheet: OracleID, LOB, LWD Expected.
#    For each EWS employee, any pivot week ON or AFTER LWD Expected
#    is overwritten with 'LG Chat Termed' or 'NL Chat Termed' based on LOB.
# ---------------------------------------------------------------------------
ews = pd.read_excel(folder_paths["hc_staffing"], sheet_name="EWS")[[
    "OracleID", "LOB", "LWD Expected", "WD"
]].dropna(subset=["OracleID", "LWD Expected"])

ews["OracleID"]     = pd.to_numeric(ews["OracleID"], errors="coerce")
ews["LWD Expected"] = pd.to_datetime(ews["LWD Expected"], errors="coerce")
ews = ews.dropna(subset=["OracleID", "LWD Expected"])

# Keep only employees still Active in Workday — INACTIVE means already
# terminated and handled upstream by the main hc_extend pipeline.
# Only Active employees need EWS forecasting until their expected LWD.
ews = ews[ews["WD"].astype(str).str.upper().str.strip() == "ACTIVE"]
ews = ews.drop(columns=["WD"])

print(f"EWS Active employees to forecast: {len(ews)}")

# Map LOB -> termed label
# Any LOB containing 'Non_Lodging' or 'NL' -> NL Chat Termed; else -> LG Chat Termed
ews["EWS_Role"] = np.where(
    ews["LOB"].str.contains("Non_Lodging|NL", na=False, regex=True),
    "NL Chat Termed",
    "LG Chat Termed",
)

# Merge EWS onto hc_monday to get LWD Expected and EWS_Role per row
hc_monday["OracleID"] = pd.to_numeric(hc_monday["OracleID"], errors="coerce")
hc_monday["Date"]     = pd.to_datetime(hc_monday["Date"], errors="coerce")

hc_monday = hc_monday.merge(
    ews[["OracleID", "LWD Expected", "EWS_Role"]],
    on="OracleID", how="left",
)

# Overwrite HC_File_Role for weeks on/after LWD Expected
ews_mask = (
    hc_monday["LWD Expected"].notna()
    & (hc_monday["Date"] >= hc_monday["LWD Expected"])
)
hc_monday.loc[ews_mask, "HC_File_Role"] = hc_monday.loc[ews_mask, "EWS_Role"]

# Drop helper columns before pivoting
hc_monday = hc_monday.drop(columns=["LWD Expected", "EWS_Role"], errors="ignore")

print(f"EWS applied: {ews_mask.sum()} rows overwritten from {ews['OracleID'].nunique()} EWS employees")

# 6. Pivot: one row per employee, one column per week
hc_pivot = pd.pivot_table(
    data=hc_monday,
    index=["OracleID", "Email Id", "Employee Name"],
    columns="Date Start Week",
    values="HC_File_Role",
    aggfunc="first",
).reset_index()
hc_pivot.columns.name = None

# 7. Export to Excel
export_folder = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount"
export_path   = f"{export_folder}/Pivot_HC_Role_2026.xlsx"
export_path2  = f"{export_folder}/HC_Role_2026.xlsx"

hc_pivot.to_excel(export_path,  index=False, engine="xlsxwriter")
hc_monday.to_excel(export_path2, index=False, engine="xlsxwriter")
print("Exported Pivot_HC_Role_2026.xlsx and HC_Role_2026.xlsx")

hc_pivot


EWS Active employees to forecast: 1
EWS applied: 4 rows overwritten from 1 EWS employees
Exported Pivot_HC_Role_2026.xlsx and HC_Role_2026.xlsx


,OracleID,Email Id,Employee Name,2026-01-05,2026-01-12,2026-01-19,2026-01-26,2026-02-02,2026-02-09,2026-02-16,...,2026-04-20,2026-04-27,2026-05-04,2026-05-11,2026-05-18,2026-05-25,2026-06-01,2026-06-08,2026-06-15,2026-06-22
0,1081503,van.tran@concentrix.com,TRAN PHAM NGOC VAN,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,...,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred,Transferred
1,101606999,hoangminhduc.nguyen@concentrix.com,NGUYEN HOANG MINH DUC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed,Leader Termed
2,101885211,thixuantien.nguyen@concentrix.com,NGUYEN THI XUAN TIEN,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,...,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed,RTA Termed
3,101957833,ducanh.nguyen@concentrix.com,NGUYEN DUC ANH,Termed,Termed,Termed,Termed,Termed,Termed,Termed,...,Termed,Termed,Termed,Termed,Termed,Termed,Termed,Termed,Termed,Termed
4,101973765,sanghdok.thanh@concentrix.com,THANH SANG HDOK,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,...,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed,LG Chat Termed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
833,103578433,thuhang.tran@concentrix.com,TRAN THU HANG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,LG Chat,LG Chat Training,LG Chat Training,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting
834,103578435,quanghung.nguyen@concentrix.com,NGUYEN QUANG HUNG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,LG Chat,LG Chat Training,LG Chat Training,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting
835,103578438,thitouyen.nguyen@concentrix.com,NGUYEN THI TO UYEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,LG Chat,LG Chat Training,LG Chat Training,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting
836,103578443,ngochien.ly@concentrix.com,LY NGOC HIEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,LG Chat,LG Chat Training,LG Chat Training,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting,LG Chat Nesting


In [ ]:
# ---------------------------------------------------------------------------
# Export one CSV per month into the HC Extend by Month folder.
# File name format: YY_MM (e.g. 25_04) for correct alphabetical sort.
#
# OPTIMISATION: before writing each monthly file, drop rows where the
# employee has been Terminated or Transferred for more than 3 months.
# Rule: if LWD/Movement is not null and the snapshot month is more than
# 3 calendar months after the LWD month, exclude that row entirely.
# Example: LWD = Jun-2025 → keep in Jun/Jul/Aug/Sep, drop from Oct onward.
# ---------------------------------------------------------------------------

hc_extend["Date"]         = pd.to_datetime(hc_extend["Date"],         errors="coerce")
hc_extend["LWD/Movement"] = pd.to_datetime(hc_extend["LWD/Movement"], errors="coerce")

for month_label, group in hc_extend.groupby("Month"):
    # Derive snapshot year/month from the first valid Date in this group
    sample_date   = pd.to_datetime(group["Date"].dropna().iloc[0], errors="coerce")
    snapshot_month = sample_date.to_period("M")   # e.g. Period('2025-10', 'M')

    # Compute how many full calendar months have elapsed since LWD/Movement
    lwd_period = group["LWD/Movement"].dt.to_period("M")
    months_since_lwd = (
        (snapshot_month.year  - lwd_period.dt.year)  * 12
        + (snapshot_month.month - lwd_period.dt.month)
    )

    # Drop rows that are termed/transferred AND LWD was > 3 months ago
    is_exited     = group["LWD/Movement"].notna()
    is_stale      = months_since_lwd > 3
    group_filtered = group[~(is_exited & is_stale)].copy()

    dropped = len(group) - len(group_filtered)
    file_name = sample_date.strftime("%y_%m") + ".csv"
    file_path = os.path.join(folder_paths["hc_extend_by_month"], file_name)
    group_filtered.to_csv(file_path, index=False)
    print(f"Written: {file_name}  |  rows: {len(group_filtered):,}  |  dropped (stale exits): {dropped:,}")


Written: 24_04.csv  |  rows: 8,821  |  dropped (stale exits): 630
Written: 25_04.csv  |  rows: 15,017  |  dropped (stale exits): 7,770
Written: 26_04.csv  |  rows: 4,443  |  dropped (stale exits): 20,310
Written: 24_08.csv  |  rows: 8,990  |  dropped (stale exits): 2,542
Written: 25_08.csv  |  rows: 9,207  |  dropped (stale exits): 15,314
Written: 23_12.csv  |  rows: 3,585  |  dropped (stale exits): 0
Written: 24_12.csv  |  rows: 13,886  |  dropped (stale exits): 4,743
Written: 25_12.csv  |  rows: 6,014  |  dropped (stale exits): 18,538
Written: 24_02.csv  |  rows: 6,614  |  dropped (stale exits): 0
Written: 25_02.csv  |  rows: 15,214  |  dropped (stale exits): 5,516
Written: 26_02.csv  |  rows: 3,820  |  dropped (stale exits): 18,396
Written: 24_01.csv  |  rows: 5,999  |  dropped (stale exits): 0
Written: 25_01.csv  |  rows: 16,825  |  dropped (stale exits): 5,363
Written: 26_01.csv  |  rows: 4,495  |  dropped (stale exits): 20,057
Written: 24_07.csv  |  rows: 9,001  |  dropped (stale

In [ ]:
# ---------------------------------------------------------------------------
# Wave Summary: group by Wave to get Training start, Nesting start,
# Go-live (= Production date), HC count at Go-live, and Pass Rate.
# Only Active employees are included (LWD/Movement is null).
# ---------------------------------------------------------------------------

wave_df = hc_extend[
    hc_extend["Wave"].notna()
    & hc_extend["Active"].str.upper().eq("YES")
].copy()

# Parse dates
for col in ["CCT Training", "Lodging Nesting", "NonLodging Nesting",
            "Lodging Production", "NonLodging Production"]:
    if col in wave_df.columns:
        wave_df[col] = pd.to_datetime(wave_df[col], errors="coerce")

# Derive unified Nesting and Production across LOB
wave_df["_Nesting"] = wave_df["Lodging Nesting"].fillna(wave_df["NonLodging Nesting"])
wave_df["_Golive"]  = wave_df["Lodging Production"].fillna(wave_df["NonLodging Production"])

# One row per unique employee per Wave (deduplicate before aggregating)
wave_unique = (
    wave_df
    .drop_duplicates(subset=["Wave", "OracleID"])
    .groupby("Wave", as_index=False)
    .agg(
        Year         = ("CCT Training",  lambda x: x.dt.year.min()),
        Month        = ("CCT Training",  lambda x: x.dt.month.min()),
        HC_Start     = ("OracleID",       "nunique"),
        Start_Training = ("CCT Training", "min"),
        Nesting      = ("_Nesting",       "min"),
        Golive       = ("_Golive",        "min"),
        HC_Golive    = ("_Golive",        lambda x: x.notna().sum()),
        LOB          = ("LOB",            lambda x: x.mode()[0] if len(x) else None),
    )
)

# Passed Rate = HC_Golive / HC_Start
wave_unique["Passed_Rate"] = (
    (wave_unique["HC_Golive"] / wave_unique["HC_Start"]).round(3) * 100
).map("{:.1f}%".format)

# Format date columns for display
for col in ["Start_Training", "Nesting", "Golive"]:
    wave_unique[col] = pd.to_datetime(wave_unique[col], errors="coerce").dt.strftime("%d-%b-%y")

wave_unique = wave_unique.sort_values(["Year", "Month", "Wave"]).reset_index(drop=True)

wave_unique


,Wave,Year,Month,HC_Start,Start_Training,Nesting,Golive,HC_Golive,LOB,Passed_Rate
0,4,2023.0,1.0,4,23-Dec-23,19-Feb-24,04-Mar-24,4,Lodging,100.0%
1,2A,2023.0,4.0,2,11-Dec-23,08-Jan-24,22-Jan-24,2,Lodging,100.0%
2,1A,2023.0,11.0,1,20-Nov-23,18-Dec-23,15-Jan-24,1,Lodging,100.0%
3,1B,2023.0,11.0,2,20-Nov-23,18-Dec-23,15-Jan-24,2,Lodging,100.0%
4,1C,2023.0,11.0,2,20-Nov-23,18-Dec-23,01-Jan-24,2,Support,100.0%
5,4,2023.0,12.0,1,23-Dec-23,19-Feb-24,04-Mar-24,1,Support,100.0%
6,5,2024.0,2.0,3,05-Feb-24,04-Mar-24,18-Mar-24,3,Lodging,100.0%
7,-,2024.0,2.0,5,23-Dec-24,NaN,09-Feb-26,2,Support,40.0%
8,7,2024.0,3.0,1,04-Mar-24,01-Apr-24,15-Apr-24,1,Lodging,100.0%
9,8,2024.0,3.0,1,18-Mar-24,15-Apr-24,29-Apr-24,1,Lodging,100.0%
